# Dev's Personalized Assistant — End-to-End Fine-Tuning

Fine-tunes a 1.5B Qwen model on Dev's labeled conversations so the agent
learns Dev-specific decision-making (when to fire a tool, what tool, how
important). The same notebook works for any persona — just change `PERSONA`
in the config cell.

**Before running:** in `My Drive/CSE534_project/` you need:
- `dev_entrepreneur_10_conversations_evolving_tool_calls_labeled.json`
- `parse_json_dataset.py`
- `train_sft.py`, `train_reward_model.py`, `train_ppo.py`, `train_dpo.py`, `eval.py`

**Runtime:** Colab Pro → A100 (recommended) or L4. T4 also works but slower.


## 1. GPU + Drive


In [ ]:
!nvidia-smi | head -20


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd '/content/drive/MyDrive/CSE534_project'
!ls


## 2. Install dependencies (~5 min)
After this cell finishes, **Runtime → Restart session** before continuing.


In [ ]:
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --upgrade 'transformers>=4.56,<5.0' 'trl>=0.20,<0.25' 'peft>=0.18' accelerate bitsandbytes datasets
!pip install -q python-docx

import torch, transformers, trl, peft
print(f'torch {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'transformers {transformers.__version__}  trl {trl.__version__}  peft {peft.__version__}')


## 3. Pick the persona & file paths

Change `PERSONA` and `JSON_FILE` for a different person — the rest of the
notebook auto-adjusts.


In [ ]:
import os

PERSONA   = 'dev'
JSON_FILE = 'dev_entrepreneur_10_conversations_evolving_tool_calls_labeled.json'
HOLDOUT   = 10        # which conversation to hold out for evaluation

os.environ['PERSONA'] = PERSONA
print(f'training assistant for: {PERSONA}')


## 4. Clean any old checkpoints (avoids the 3B/1.5B shard collision)


In [ ]:
!rm -rf {PERSONA}-sft {PERSONA}-sft-merged
!rm -rf {PERSONA}-rm  {PERSONA}-rm-final
!rm -rf {PERSONA}-dpo {PERSONA}-dpo-merged
!rm -rf {PERSONA}-ppo-final
!ls | grep -i ${PERSONA} || echo 'clean.'


## 5. Parse the persona JSON

Produces `{persona}_sft.jsonl`, `{persona}_dpo.jsonl`, `{persona}_test.jsonl`.
The persona's traits from the JSON's `assumptions` field get baked into the
system prompt so the model learns this person's specific decision style.


In [ ]:
!python parse_json_dataset.py {JSON_FILE} {PERSONA} --holdout {HOLDOUT}
!cat {PERSONA}_stats.json


## 6. SFT — supervised baseline (~10 min on A100)


In [ ]:
!python train_sft.py


## 7. Reward model — RLHF branch (~5 min)


In [ ]:
!python train_reward_model.py


## 8. PPO — RLHF branch (~30 min on A100)

Tighter KL (0.15) than the previous run to keep the policy on-distribution.
If it OOMs, drop `response_length` to 32 in `train_ppo.py`.


In [ ]:
!python train_ppo.py


## 9. DPO — alternative branch (~10 min)


In [ ]:
!python train_dpo.py


## 10. Evaluate all four models on Dev's held-out conversation


In [ ]:
!python eval.py


## 11. Quick sanity check — what does the trained model say to a fresh utterance?

Type something Dev might say and watch each model decide.


In [ ]:
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = f'./{PERSONA}-dpo-merged'  # try -sft-merged or -ppo-final too
tok   = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.bfloat16, device_map='auto').eval()

with open(f'{PERSONA}_test.jsonl') as f:
    sample = json.loads(f.readline())
system = sample['system']

def ask(user):
    msgs = [{'role':'system','content':system}, {'role':'user','content':user}]
    p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(p, return_tensors='pt').to(model.device)
    out = model.generate(**inp, max_new_tokens=80, do_sample=False, pad_token_id=tok.pad_token_id)
    print(f'{user!r}\n  -> {tok.decode(out[0, inp.input_ids.shape[1]:], skip_special_tokens=True).strip()}')

ask('Block 9 to 11 tomorrow morning for the Patel proposal deep-work block.')
ask('Remind me to approve the packaging vendor advance after the proposal block.')
ask('How was your weekend?')
ask('Set an alarm for 2:50 PM so I do not miss the investor call prep.')
ask('What is the capital of France?')


## 12. Pick the winning model & convert to GGUF for Ollama

Replace `PICK` with whichever performed best in step 10.


In [ ]:
PICK = f'{PERSONA}-dpo-merged'   # or f'{PERSONA}-ppo-final', etc.

!git clone -q https://github.com/ggerganov/llama.cpp 2>/dev/null || true
!pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py ./{PICK} \
    --outfile {PERSONA}-assistant.gguf --outtype q4_k_m
!ls -lh {PERSONA}-assistant.gguf


## 13. Save artifacts back to Drive

Everything is already on Drive (you `cd`'d into a Drive folder), so the
`.gguf` and the merged checkpoints persist. Optional: zip the winning model
for easier download to your Mac.


In [ ]:
!zip -qr {PICK}.zip {PICK}
!ls -lh {PICK}.zip {PERSONA}-assistant.gguf
